In [3]:
##plotting when was the best time to make jokes using visualization 2-defined function
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import scipy.stats as stats
from math import floor
import matplotlib.pyplot as plt
import plotly.express as px 
from pathlib import Path
from matplotlib.image import imread
import json
from datetime import datetime, timedelta
import nbimporter
import plotly.express as px
from scipy.ndimage import gaussian_filter1d
import plotly.graph_objects as go
# Get path to the project root: project/
PROJECT_ROOT = Path.cwd().parents[1]

# Add src/ to Python path
SRC_DIR = PROJECT_ROOT / "src"

sys.path.append(str(SRC_DIR))


#for theme
%run ../../docs/plotly_theme.ipynb


In [4]:

FIGURES_DIR = PROJECT_ROOT / 'figures'

# DATA DIRECTORIES
DATA_DIR = PROJECT_ROOT / 'data' / 'newyorker_caption_contest'
JSON_DIR = DATA_DIR / 'contests.json'
CSV_DIR = DATA_DIR / 'data'
CSV_CLEAN_DIR = DATA_DIR / 'data_cleaned'
IMG_DIR = DATA_DIR / 'images'

print("Project path setup:")
print("PROJECT_ROOT:  ", PROJECT_ROOT)
print("SRC DIR     :" , SRC_DIR)
print("FIGURES_DIR:   ", FIGURES_DIR)
print("\nDATA_DIR:      ", DATA_DIR)
print("JSON_DIR:      ", JSON_DIR)
print("CSV_DIR:       ", CSV_DIR)
print("CSV_CLEAN_DIR: ", CSV_CLEAN_DIR)
print("IMG_DIR:       ", IMG_DIR)

Project path setup:
PROJECT_ROOT:   c:\Users\hanna\ADA\ada-2025-project-json-derulo
SRC DIR     : c:\Users\hanna\ADA\ada-2025-project-json-derulo\src
FIGURES_DIR:    c:\Users\hanna\ADA\ada-2025-project-json-derulo\figures

DATA_DIR:       c:\Users\hanna\ADA\ada-2025-project-json-derulo\data\newyorker_caption_contest
JSON_DIR:       c:\Users\hanna\ADA\ada-2025-project-json-derulo\data\newyorker_caption_contest\contests.json
CSV_DIR:        c:\Users\hanna\ADA\ada-2025-project-json-derulo\data\newyorker_caption_contest\data
CSV_CLEAN_DIR:  c:\Users\hanna\ADA\ada-2025-project-json-derulo\data\newyorker_caption_contest\data_cleaned
IMG_DIR:        c:\Users\hanna\ADA\ada-2025-project-json-derulo\data\newyorker_caption_contest\images


In [5]:
def add_contest_dates(df):
    """
    Adds publish and vote dates for each contest based on weekly frequency.
    Contest 510 is assumed to be published on 22 Feb 2016.
    """

    base_publish = datetime.strptime('2016.02.22', "%Y.%m.%d")

    publish_dates = []
    vote_dates = []

    for i in range(len(df)):
        contest_id = df['contest_id'].iloc[i]
        diff = int(contest_id) - 510

        publish_date = base_publish + timedelta(days=diff * 7)
        vote_date = publish_date + timedelta(days=7)

        # Ensure metadata is a dictionary
        if df.at[i, 'metadata'] is None:
            df.at[i, 'metadata'] = {}

        # Now safe to assign
        df.at[i, 'metadata']['contest_day_publish'] = publish_date
        df.at[i, 'metadata']['contest_day_votes'] = vote_date

        publish_dates.append(publish_date)
        vote_dates.append(vote_date)

    df['contest_day_publish'] = publish_dates
    df['contest_day_votes'] = vote_dates

    return df

In [6]:
contest_no_dates = pd.read_json(JSON_DIR)
contest_with_dates = add_contest_dates(contest_no_dates)
contest_with_dates

,contest_id,image,data,metadata,contest_day_publish,contest_day_votes
0,510,images/510.jpg,data/510.csv,"{'num_captions': 3905, 'num_votes': 41185, 'im...",2016-02-22,2016-02-29
1,511,images/511.jpg,data/511.csv,"{'num_captions': 3325, 'num_votes': 28205, 'im...",2016-02-29,2016-03-07
2,512,images/512.jpg,data/512.csv,"{'num_captions': 4399, 'num_votes': 21574, 'im...",2016-03-07,2016-03-14
3,513,images/513.jpg,data/513.csv,"{'num_captions': 4141, 'num_votes': 16894, 'im...",2016-03-14,2016-03-21
4,514,images/514.jpg,data/514.csv,"{'num_captions': 3951, 'num_votes': 95790, 'im...",2016-03-21,2016-03-28
...,...,...,...,...,...,...
379,891,images/891.jpg,data/891.csv,"{'contest_day_publish': 2023-06-12 00:00:00, '...",2023-06-12,2023-06-19
380,892,images/892.jpg,data/892.csv,"{'contest_day_publish': 2023-06-19 00:00:00, '...",2023-06-19,2023-06-26
381,893,images/893.jpg,data/893.csv,"{'contest_day_publish': 2023-06-26 00:00:00, '...",2023-06-26,2023-07-03
382,894,images/894.jpg,data/894.csv,"{'contest_day_publish': 2023-07-03 00:00:00, '...",2023-07-03,2023-07-10


NOW AS PLOTLY

We can see clearly the peak due to the pandemic effect in the number of submitted caption.
More people were online

More free time

Massive spike in all digital engagement

Caption contests, puzzles, games all saw surges

In [11]:

def plot_captions_and_funniness_timeline_plotly(
    folder_path,
    contest_with_dates,
    sigma
):
    folder_path = str(folder_path)
    records = []

    # --- Loop over contests ---
    for contest_num in range(510, 896):
        filename = f"{contest_num}_cleaned.csv"
        file_path = os.path.join(folder_path, filename)

        if not os.path.exists(file_path):
            continue

        try:
            df = pd.read_csv(file_path)

            records.append({
                "Contest": contest_num,
                "Num_Captions": len(df),
                "Mean_Funniness": df["mean"].mean()
            })

        except Exception as e:
            print(f"Error reading {filename}: {e}")

    # --- Create DataFrame ---
    df_metrics = pd.DataFrame(records)

    # --- Merge with publish dates ---
    merged = (
        df_metrics
        .merge(
            contest_with_dates[["contest_id", "contest_day_publish"]],
            left_on="Contest",
            right_on="contest_id",
            how="inner"
        )
        .sort_values("contest_day_publish")
    )

    # --- Gaussian smoothing ---
    merged["smooth_captions"] = gaussian_filter1d(
        merged["Num_Captions"], sigma=sigma
    )
    merged["smooth_funniness"] = gaussian_filter1d(
        merged["Mean_Funniness"], sigma=sigma
    )

    # --- Build Plotly figure ---
    fig = go.Figure()

    # ===== Y1: Number of captions =====
    fig.add_trace(go.Scatter(
        x=merged["contest_day_publish"],
        y=merged["Num_Captions"],
        mode="markers",
        name="Captions (raw)",
        opacity=0.5,
        marker=dict(size=6),
        yaxis="y1",
        customdata=merged["Contest"],
        hovertemplate=(
            "<b>Contest</b>: %{customdata}<br>"
            "<b>Number of Captions</b>: %{y}"
            "<extra></extra>"
        )

        
    ))

    fig.add_trace(go.Scatter(
        x=merged["contest_day_publish"],
        y=merged["smooth_captions"],
        mode="lines",
        line=dict(color="darkgoldenrod", width=3),
        name=f"Captions (smooth) (σ={sigma})",
        yaxis="y1",
        hovertemplate=(
            "<b>Smoothed number of captions</b>: %{y:.2f}<extra></extra>"
        )
    ))

    # ===== Y2: Mean funniness =====
    fig.add_trace(go.Scatter(
        x=merged["contest_day_publish"],
        y=merged["Mean_Funniness"],
        mode="markers",
        name="Mean funniness (raw)",
        opacity=0.5,
        marker=dict(size=6, color="firebrick"),
        yaxis="y2",
        hovertemplate=(
            "<b>Mean funniness</b>: %{y:.2f}<extra></extra>"
        )
    ))

    fig.add_trace(go.Scatter(
        x=merged["contest_day_publish"],
        y=merged["smooth_funniness"],
        mode="lines",
        line=dict(color="firebrick", width=3),
        name=f"Mean funniness (smooth) (σ={sigma})",
        yaxis="y2",
        hovertemplate=(
            "<b>Smoothed mean funniness</b>: %{y:.2f}<extra></extra>"
        )
    ))

    # --- Layout ---
    fig.update_layout(
        title="Contest Popularity and Mean Funniness Over Time",
        xaxis=dict(
            title="Publish Date",
            rangeslider=dict(visible=False)  # ← THIS LINE
        ),
        yaxis=dict(
            title="Number of Captions",
            side="left"
        ),
        yaxis2=dict(
            title="Mean Funniness",
            overlaying="y",
            side="right",
            rangemode="normal"
        ),
        template="for_dummies",
        hovermode="x unified",
        width=740,
        height=600
    )
    # --- Save and show ---
    save_path = (
        r"C:\Users\hanna\ADA\ada-2025-project-json-derulo"
        r"\docs\plots\captions_funniness_timeline_plot.html"
    )

    fig.write_html(save_path, include_plotlyjs="cdn")
    fig.show()

    return merged

In [12]:
plot_captions_and_funniness_timeline_plotly(CSV_CLEAN_DIR, contest_with_dates, sigma=4)

,Contest,Num_Captions,Mean_Funniness,contest_id,contest_day_publish,smooth_captions,smooth_funniness
0,510,3905,1.090060,510,2016-02-22,4234,1.150079
1,511,3325,1.110889,511,2016-02-29,4283,1.152573
2,512,4399,1.121523,512,2016-03-07,4376,1.157219
3,513,4141,1.178147,513,2016-03-14,4501,1.163414
4,514,3951,1.296701,514,2016-03-21,4645,1.170453
...,...,...,...,...,...,...,...
379,891,5159,1.183106,891,2023-06-12,5542,1.191325
380,892,5188,1.205958,892,2023-06-19,5540,1.190927
381,893,5513,1.218253,893,2023-06-26,5539,1.190607
382,894,6157,1.169975,894,2023-07-03,5538,1.190382
